# H3 Video-Prompt Writer

Writes properly-formatted H3 video-generation prompts (T2VA / I2VA / FL2VA / L2VA) using a local LLM running in this Colab session.

**How it works:**
1. This notebook downloads a quantized Qwen3-14B model and the official H3 prompt-writing guide from Hugging Face on every run (no account/credentials needed — Colab has no persistent disk, so this happens every session).
2. It launches a Gradio web app where you describe what you want — fix a rough prompt, combine two prompts, remix several, add a camera motion, generate from scratch, anything — pick a mode, and get back a correctly formatted H3 prompt with a copy button.
3. Every input/output pair is logged locally with an id and timestamp. Use the "Download history" button in the app before your Colab session ends if you want to keep it — it does not persist on its own.

**Requirements:** a GPU runtime. Runtime -> Change runtime type -> T4 GPU (free tier is sufficient).

Run the two cells below in order.

---

Maintained at [github.com/citronlegacy/prompt_writer_collab](https://github.com/citronlegacy/prompt_writer_collab) by Citron Legacy.

In [ ]:
# Cell 2: Install and setup
# Installs dependencies (CUDA-enabled llama-cpp-python + Gradio + huggingface_hub)
# and downloads the model and guide anonymously — no Hugging Face login required.

!pip install -q gradio huggingface_hub

# Install a prebuilt CUDA wheel for llama-cpp-python instead of compiling from
# source (a from-source CUDA build can take 15-30+ minutes in Colab). Falls
# back through a couple of recent CUDA tags to match whatever Colab ships.
import subprocess

cuda_tag = "cu124"
try:
    nvcc_output = subprocess.run(
        ["nvcc", "--version"], capture_output=True, text=True, check=True
    ).stdout
    if "release 12.1" in nvcc_output:
        cuda_tag = "cu121"
    elif "release 12.2" in nvcc_output:
        cuda_tag = "cu122"
    elif "release 12.5" in nvcc_output or "release 12.6" in nvcc_output:
        cuda_tag = "cu125"
except Exception:
    pass  # keep the cu124 default, the current Colab base image as of writing

wheel_index = f"https://abetlen.github.io/llama-cpp-python/whl/{cuda_tag}"
print(f"Installing llama-cpp-python prebuilt wheel for {cuda_tag}...")
!pip install -q llama-cpp-python --extra-index-url {wheel_index}

import os
from huggingface_hub import hf_hub_download

os.makedirs("/content", exist_ok=True)

model_path = hf_hub_download(
    repo_id="mradermacher/Qwen3-14B-abliterated-GGUF",
    filename="Qwen3-14B-abliterated.Q4_K_M.gguf",
    local_dir="/content",
)

guide_path = hf_hub_download(
    repo_id="MiniMaxAI/MiniMax-H3",
    filename="docs/VIDEO_PROMPT_WRITING_GUIDE_base_en.md",
    local_dir="/content",
)
# hf_hub_download preserves the repo's subfolder structure; make sure the
# guide ends up at the flat path app.py expects.
flat_guide_path = "/content/VIDEO_PROMPT_WRITING_GUIDE_base_en.md"
if guide_path != flat_guide_path:
    import shutil
    shutil.copyfile(guide_path, flat_guide_path)

print("Model downloaded to:", model_path)
print("Guide downloaded to:", flat_guide_path)

# Pull this project's own source files (app.py, system_prompt.py, example bank)
!rm -rf /content/prompt_writer_collab_src
!git clone --depth 1 https://github.com/citronlegacy/prompt_writer_collab /content/prompt_writer_collab_src
import sys
sys.path.insert(0, "/content/prompt_writer_collab_src")

In [ ]:
# Cell 3: Run the app
# Launches the Gradio UI with a public share link. A random username/password
# is generated and printed below — use it to log into the app URL.

from app import launch

launch()